<span style='color:#0066cc'> <span style='font-family:serif'> <font size="15"> **Access Reanalysis Data From MERRA2**

MERRA-2 is the latest version of global atmospheric reanalysis for the satellite era produced by NASA Global Modeling and Assimilation Office (GMAO) using the Goddard Earth Observing System Model (GEOS) version 5.12.4. The dataset covers the period of 1980-present with the latency of ~3 weeks after the end of a month. 

`M2I3NPASM` (or inst3_3d_asm_Np) is an instantaneous 3-dimensional 3-hourly data collection in Modern-Era Retrospective analysis for Research and Applications version 2 (MERRA-2). This collection consists of assimilations of meteorological parameters at 42 pressure levels, such as temperature, wind components, vertical pressure velocity, water vapor, ozone mass mixing ratio, and layer height. **Source**: [NASA Earthdata](https://www.earthdata.nasa.gov/data/catalog/ges-disc-m2i3npasm-5.12.4#toc-copy-citation)

`M2I3NVCHM` (or inst3_3d_chm_Nv) is an instantaneous 3-dimensional 3-hourly data collection in Modern-Era Retrospective analysis for Research and Applications version 2 (MERRA-2). This collection consists of assimilations of carbon monoxide and ozone mixing ratio at 72 model layers. **Source:** [NASA Earthdata](https://www.earthdata.nasa.gov/data/catalog/ges-disc-m2i3nvchm-5.12.4).

 <span style='color:#ff6666'><font size="5"> **Requirements**
1. <font size="3"> Valid Earthdata Login (EDL)

 <span style='color:#ff6666'><font size="5"> **Objectives**

### Subset a remote file

- <font size="3">**a)** By Variables
- <font size="3">**b)** By Spatial selection

### Subset multiple remote files

- <font size="3">Stream subset of data

<span style='color:#ff6666'><font size="5"> **References**

<font size="3"> **Global Modeling And Assimilation Office, &amp; Pawson, S. (2015)**. <i>MERRA-2 inst3_3d_asm_Np: 3d,3-Hourly,Instantaneous,Pressure-Level,Assimilation,Assimilated Meteorological Fields V5.12.4</i> [Data set]. NASA Goddard Earth Sciences Data and Information Services Center. https://doi.org/10.5067/QBZ6MG944HW0

<font size="3"> **Global Modeling And Assimilation Office, &amp; Pawson, S. (2015)**. <i>MERRA-2 inst3_3d_chm_Nv: 3d,3-Hourly,Instantaneous,Model-Level,Assimilation,Carbon Monoxide and Ozone Mixing Ratio V5.12.4</i> [Data set]. NASA Goddard Earth Sciences Data and Information Services Center. https://doi.org/10.5067/HO9OVZWF3KW2

In [ ]:
import xarray as xr
import datetime as dt
import earthaccess
import numpy as np
import matplotlib.pyplot as plt
# import pydap-specific tools
from pydap.client import open_url, get_cmr_urls 
from pydap.client import to_netcdf as dap_to_netcdf

# Finding OPeNDAP URLs

<span style='font-family:serif'> <font size="5.5"><span style='color:#0066cc'> **Query opendap urls using NASA's CMR API**    
    
We are interested in `MERRA-2` data, hourly data. MERRA-2 has several collections, organized by data products and frequency of data. In this case, we are interested in hourly data from the following collections:
    
* **doi**: 10.5067/QBZ6MG944HW0, **shortname**: M2I3NPASM. **Variables of interest**: air_temperature, surface_pressure, Ertel's Potential Vorticity, northward_wind, eastward_wind, specific_humidity, ozone_mass_mixing_ratio and (dimensions). Further information [click here]( https://disc.gsfc.nasa.gov/datasets/M2I3NPASM_5.12.4/summary).

* **doi**: 10.5067/HO9OVZWF3KW2, **shortname**: M2I3NVCHM, **Variables of interest**: Carbon Monoxide, O3 (and dimensions data). Further information [click here](https://disc.gsfc.nasa.gov/datasets/M2I3NVCHM_5.12.4/summary).


In [ ]:
MERRA2_M2I3NPASM_doi = "10.5067/QBZ6MG944HW0" # 
MERRA2_M2I3NPASM_ccid = "C1276812879-GES_DISC"

MERRA2_M2I3NVCHM_doi = "10.5067/HO9OVZWF3KW2"
MERRA2_M2I3NVCHM_ccid = "C1276812901-GES_DISC"

year_start = 2024
year_end = 2025


time_ranges = [[dt.datetime(year, 9, 21), dt.datetime(year, 10, 22)] for year in range(year_start, year_end)]

cmr_urls1 = [urls for time in time_ranges for urls in get_cmr_urls(doi=MERRA2_M2I3NPASM_doi, time_range=time, limit=1000)] # you can incread the limit of results
cmr_urls2 = [urls for time in time_ranges for urls in get_cmr_urls(doi=MERRA2_M2I3NVCHM_doi, time_range=time, limit=1000)]

print("################################################ \n We found a total of ", len(cmr_urls1)+len(cmr_urls2), "OPeNDAP URLS!!!\n################################################")

In [ ]:
len(cmr_urls1), len(cmr_urls2)

<span style='font-family:serif'> <font size="5.5"><span style='color:#0066cc'> **EDL Authentication via earthaccess and OPeNDAP**

<font size="3.5"> You can authenticate via earthaccess as demonstrated below. You must have a valid EDL account. There are two strategies for authenticating with `earthaccess`:

1. `strategy="interactive"`. This will promt your edl username-password.
2. `strategy="netrc"`. Use this if the notebook is running on an environment where a `.netrc` with your credentials is recoverable. T

<font size="3.5"> Below the default will be `netrc`, assuming the user has executed the notebook **Authenticate.ipynb**. If not, you can change the strategy to `"interactive"`.



In [ ]:
from earthaccess.exceptions import LoginStrategyUnavailable
try:
    auth = earthaccess.login(strategy="netrc", persist=True) # you will be promted to add your EDL credentials
except LoginStrategyUnavailable:
    auth = earthaccess.login(strategy="interactive", persist=True)

# pass Token Authorization to a new Session.
my_session = session=auth.get_session()


<span style='font-family:serif'> <font size="5.5"><span style='color:#0066cc'> **Accessing Metadata-ONLY with PyDAP**

<font size="3"> We can access <span style='color:#ff6666'>**OPeNDAP**</span>-produced metadata to identify the variables of interest. In particular those associated with latitude and longitude values

<font size="3"> Below need to request the <span style='color:#ff6666'>**DAP4**</span> metadata from the remote server. To specify the protocol, we have 2 options:

**1.** <font size="3"> Replace "https" with "dap4" in the URL. This works when using `Xarray`:
```python
open_url(url.replace("https","dap4"), ...)
```
**2**. <font size="3"> Specify the protocol directly (**not compatible with Xarray**)
```python
open_url(url, protocol='dap4', ...)
```

<font size="3"> Below we follow option **2)** above.


In [ ]:
%%time
pyds1 = open_url(cmr_urls1[0], protocol="dap4", session=my_session)
pyds1.tree()

In [ ]:
%%time
pyds2 = open_url(cmr_urls2[0], protocol="dap4", session=my_session)
pyds2.tree()

In [ ]:
# output data directory
output_path = "data/"

Variables1 = ["/U", "/V", "/T","/PS"]
dims1 = list(set([dim for var in pyds1 for dim in pyds1[var].dims]))
Variables1 += dims1
print("Download only Vars in 1: ", Variables1)

Variables2 = ["/CO", "/O3"]
dims2 = list(set([dim for var in pyds2 for dim in pyds2[var].dims]))
Variables2 += dims2
print("Download only Vars in 2: ", Variables2)


## Subset by lat/lon regions


<span style='font-family:serif'><font size="3.5"> In addition to subsetting by Variable names, We are interested in a region of the NorthAtlantic Ocean delimited by:


In [ ]:
lat_min, lat_max = 15, 65
lon_min, lon_max = -75, 15

## Load coordinate data 

We only need one single file per collection.

In [ ]:
%%time
ds1 = xr.open_dataset(cmr_urls1[0].replace("https", "dap4"), engine="pydap", session=my_session)

In [ ]:
%%time
ds2 = xr.open_dataset(cmr_urls2[0].replace("https", "dap4"), engine="pydap", session=my_session)

### check that coordinate data is equalacross collections

<span style='font-family:serif'><font size="3.5"> This is a necessary step, to check if same subset can be applied to data from both collections


In [ ]:
Lon1, Lat1 = ds1['lon'], ds1['lat']
Lon2, Lat2 = ds2['lon'], ds2['lat']


iLon1 = np.where((Lon1>lon_min)&(Lon1 < lon_max))[0]
iLon2 = np.where((Lon2>lon_min)&(Lon2 < lon_max))[0]

iLat1 = np.where((Lat1>lat_min)&(Lat1 < lat_max))[0]
iLat2 = np.where((Lat2>lat_min)&(Lat2 < lat_max))[0]

if all(iLon1==iLon2) and all(iLat1==iLat2):
    iLon = iLon1
    iLat = iLat2
    print("Both collections belong to same Level 3 model output")
else:
    print('Data is not Level 3')


In [ ]:
dim_slices = {
    'lon': (iLon[0], iLon[-1]), 
    'lat': (iLat[0], iLat[-1]),
}

# Stream data into local directory

In [ ]:
%%time
dap_to_netcdf(cmr_urls1, session=my_session, keep_variables=Variables1, dim_slices=dim_slices, output_path=output_path)

In [ ]:
%%time
dap_to_netcdf(cmr_urls2, session=my_session, keep_variables=Variables2, dim_slices=dim_slices, output_path=output_path)

## Inspect the data

In [ ]:
ds1 = xr.open_mfdataset(output_path+"/MERRA2_400.inst3_3d_asm_*.nc4")
ds1

In [ ]:
ds2 = xr.open_mfdataset(output_path+"/MERRA2_400.inst3_3d_chm_*.nc4")
ds2